# 02 Train and compare the retriever

Cell-by-cell replay: scope filter, source-document holdout, TF-IDF baseline, frozen MiniLM, fine-tuning, and metrics. A new run directory is used; run 001 and 002 remain unchanged. This is exploratory training on unreviewed candidate answers, not medical validation.

In [1]:
from pathlib import Path
import json,re
import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupShuffleSplit
ROOT=Path.cwd()
if not (ROOT/'data').exists(): ROOT=ROOT.parent
RUN_NAME='medquad_minilm_scope_docsplit_notebook_v003'
OUTPUT_DIR=ROOT/'reports/models'/RUN_NAME
assert not OUTPUT_DIR.exists(), f'{OUTPUT_DIR} already exists. Choose a new RUN_NAME to preserve old results.'
BASE_MODEL='sentence-transformers/all-MiniLM-L6-v2'
SEED,EPOCHS=42,4
print('Output:',OUTPUT_DIR,'| device:','cuda' if torch.cuda.is_available() else 'cpu')

Output: /home/hamse/alu/ml/diabetes-education-qa/reports/models/medquad_minilm_scope_docsplit_notebook_v003 | device: cpu


In [2]:
queue=pd.read_csv(ROOT/'data/review/medquad_review_queue.csv').fillna('')
scope=pd.read_csv(ROOT/'data/review/medquad_scope_screen.csv').fillna('')
allowed=set(scope.loc[scope.scope_decision_draft.eq('in_scope_pending_content_review'),'qa_id'])
rows=queue[queue.qa_id.isin(allowed)].copy()
rows=rows[rows.question.str.strip().ne('') & rows.answer.str.strip().ne('')].reset_index(drop=True)
print(f'{len(rows)} candidates across {rows.doc_id.nunique()} documents')
rows[['qa_id','question','answer','doc_id']].head(3)

156 candidates across 35 documents


,qa_id,question,answer,doc_id
0,0000052-1,What is (are) Diabetic Heart Disease ?,"The term ""diabetic heart disease"" (DHD) refers...",52
1,0000052-2,What causes Diabetic Heart Disease ?,"At least four complex processes, alone or comb...",52
2,0000052-3,Who is at risk for Diabetic Heart Disease? ?,People who have type 1 or type 2 diabetes are ...,52


In [3]:
def normalize(text): return re.sub(r'\W+',' ',str(text).casefold()).strip()
def retrieval_metrics(rankings,relevant_ids):
    r1=[]; r3=[]; rr=[]
    for ranked,relevant in zip(rankings,relevant_ids):
        relevant=set(relevant); r1.append(float(bool(set(ranked[:1])&relevant))); r3.append(float(bool(set(ranked[:3])&relevant)))
        rank=next((i+1 for i,x in enumerate(ranked) if x in relevant),None); rr.append(0. if rank is None else 1./rank)
    return {'recall_at_1':float(np.mean(r1)),'recall_at_3':float(np.mean(r3)),'mrr':float(np.mean(rr)),'n_queries':len(relevant_ids)}
rows['_index']=np.arange(len(rows)); rows['_qnorm']=rows.question.map(normalize)
groups=sorted(rows.doc_id.unique())
train_idx,test_idx=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=SEED).split(np.arange(len(groups)),groups=groups))
train_docs,test_docs={groups[i] for i in train_idx},{groups[i] for i in test_idx}
test_rows=rows[rows.doc_id.isin(test_docs)].copy(); test_questions=set(test_rows._qnorm)
train_rows=rows[rows.doc_id.isin(train_docs)&~rows._qnorm.isin(test_questions)].copy(); train_questions=set(train_rows._qnorm)
print('Train/test documents:',len(train_docs),len(test_docs),'| queries:',len(train_questions),len(test_questions))
assert not train_docs.intersection(test_docs) and not train_questions.intersection(test_questions)

Train/test documents: 28 7 | queries: 86 13


In [4]:
answers_by_query=train_rows.groupby('_qnorm').answer.nunique()
unambiguous=set(answers_by_query[answers_by_query.eq(1)].index)
train_pairs=train_rows[train_rows._qnorm.isin(unambiguous)].drop_duplicates('_qnorm')[['question','answer']].values.tolist()
test_query_list=[]; relevant_ids=[]
for qnorm in sorted(test_questions):
    matching=test_rows[test_rows._qnorm.eq(qnorm)]
    test_query_list.append(matching.iloc[0].question); relevant_ids.append(matching._index.tolist())
corpus=rows.answer.tolist()
print('Pairs:',len(train_pairs),'| test queries:',len(test_query_list))
pd.DataFrame(train_pairs,columns=['question','positive_answer']).head()

Pairs: 61 | test queries: 13


,question,positive_answer
0,What causes Diabetic Heart Disease ?,"At least four complex processes, alone or comb..."
1,Who is at risk for Diabetic Heart Disease? ?,People who have type 1 or type 2 diabetes are ...
2,What are the symptoms of Diabetic Heart Disease ?,Some people who have diabetic heart disease (D...
3,How to diagnose Diabetic Heart Disease ?,Your doctor will diagnose diabetic heart disea...
4,What are the treatments for Diabetic Heart Dis...,Diabetic heart disease (DHD) is treated with l...


In [5]:
tfidf=TfidfVectorizer(ngram_range=(1,2),min_df=1,sublinear_tf=True)
doc_vectors=tfidf.fit_transform(corpus); query_vectors=tfidf.transform(test_query_list)
tfidf_scores=(query_vectors@doc_vectors.T).toarray()
tfidf_result=retrieval_metrics([list(np.argsort(-s)) for s in tfidf_scores],relevant_ids)
pd.DataFrame([tfidf_result],index=['TF-IDF baseline'])

,recall_at_1,recall_at_3,mrr,n_queries
TF-IDF baseline,0.153846,0.384615,0.304767,13


In [6]:
from sentence_transformers import SentenceTransformer
device='cuda' if torch.cuda.is_available() else 'cpu'
model=SentenceTransformer(BASE_MODEL,device=device)
def rank_with_model(m):
    q=m.encode(test_query_list,convert_to_numpy=True,normalize_embeddings=True,show_progress_bar=False)
    a=m.encode(corpus,convert_to_numpy=True,normalize_embeddings=True,show_progress_bar=False)
    return [list(np.argsort(-s)) for s in q@a.T]
frozen_result=retrieval_metrics(rank_with_model(model),relevant_ids)
pd.DataFrame([frozen_result],index=['Frozen MiniLM'])

/tmp/diabetes-qa-venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3278.39it/s]

,recall_at_1,recall_at_3,mrr,n_queries
Frozen MiniLM,0.230769,0.461538,0.391866,13


## Fine-tune

The next cell trains for four epochs. Epoch logging prints progress as each epoch finishes. Hyperparameters and loss match the prior document-split experiment.

In [7]:
from datasets import Dataset
from sentence_transformers import SentenceTransformerTrainer,SentenceTransformerTrainingArguments
from sentence_transformers.sentence_transformer.losses import MultipleNegativesRankingLoss
OUTPUT_DIR.mkdir(parents=True,exist_ok=False)
train_dataset=Dataset.from_dict({'anchor':[x[0] for x in train_pairs],'positive':[x[1] for x in train_pairs]})
args=SentenceTransformerTrainingArguments(output_dir=str(OUTPUT_DIR/'trainer_state'),num_train_epochs=EPOCHS,per_device_train_batch_size=8,learning_rate=2e-5,warmup_steps=.1,weight_decay=.01,fp16=False,bf16=False,use_cpu=(device=='cpu'),save_strategy='no',logging_strategy='epoch',report_to='none',seed=SEED,dataloader_num_workers=0,optim='adamw_torch')
trainer=SentenceTransformerTrainer(model=model,args=args,train_dataset=train_dataset,loss=MultipleNegativesRankingLoss(model))
train_result=trainer.train()
print('Training loss:',train_result.training_loss)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
8,0.774169
16,0.454357
24,0.309264
32,0.158841


Training loss: 0.42415789887309074


In [8]:
fine_tuned_result=retrieval_metrics(rank_with_model(model),relevant_ids)
results={'tfidf_word_ngrams':tfidf_result,'minilm_frozen':frozen_result,'minilm_finetuned':fine_tuned_result}
pd.DataFrame(results).T

,recall_at_1,recall_at_3,mrr,n_queries
tfidf_word_ngrams,0.153846,0.384615,0.304767,13.0
minilm_frozen,0.230769,0.461538,0.391866,13.0
minilm_finetuned,0.461538,0.615385,0.586993,13.0


In [9]:
model_dir=OUTPUT_DIR/'model'; model.save(str(model_dir))
report={'data_status':'provisional_unreviewed','base_model':BASE_MODEL,'seed':SEED,'device':device,'input_rows':len(rows),'unique_queries':len(set(rows._qnorm)),'train_query_count':len(train_questions),'test_query_count':len(test_questions),'train_document_count':len(train_docs),'test_document_count':len(test_docs),'split_strategy':'doc_id','scope_draft_only':True,'contrastive_training_pairs':len(train_pairs),'epochs':EPOCHS,'train_loss':train_result.training_loss,'results':results,'notes':'Candidate answers are unreviewed; experiment is not medical validation.'}
(OUTPUT_DIR/'results.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
pd.DataFrame(results).T.to_csv(OUTPUT_DIR/'retrieval_results.csv')
print('Saved model:',model_dir,'\nSaved metrics:',OUTPUT_DIR/'results.json')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  8.90it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  8.78it/s]

Saved model: /home/hamse/alu/ml/diabetes-education-qa/reports/models/medquad_minilm_scope_docsplit_notebook_v003/model 
Saved metrics: /home/hamse/alu/ml/diabetes-education-qa/reports/models/medquad_minilm_scope_docsplit_notebook_v003/results.json


## Interpretation

The held-out set is small, so one query can shift the metrics substantially. Inspect errors; ranking scores do not establish clinical accuracy or safety.